# 41 — Cross Reference Existing Outputs

Cross-references the existing AQ26 outputs against the configured site registry.
This notebook is tolerant of missing upstream outputs and produces a unified daily table.

In [ ]:
import os, json, hashlib, platform, sys, math, time
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "41_cross_reference_existing_outputs"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

registry_path = OUTPUTS / "40_target_control_extension" / "sites_registry.csv"
registry, registry_meta = safe_read_csv(registry_path)
if registry.empty:
    registry = sites_df.copy()

dates = pd.date_range(date_from, date_to, freq="D")
base_daily = pd.DataFrame({"date": dates})

weather_candidates = [
    "outputs/05_metoffice_datahub_weather/weather_hourly.csv",
    "outputs/newhaven_fusion/weather_hourly.csv",
    "outputs/32_newhaven_fusion/weather_hourly.csv",
]
ground_candidates = [
    "outputs/04_ground_aq_providers/ground_daily.csv",
    "outputs/newhaven_fusion/ground_data_daily.csv",
    "outputs/32_newhaven_fusion/ground_data_daily.csv",
]
news_candidates = [
    "outputs/03_news_context/news_articles.json",
    "outputs/newhaven_fusion/news_articles.csv",
    "outputs/32_newhaven_fusion/news_articles.csv",
]

weather = pd.DataFrame({"date": dates})
weather_meta = []
for cand in weather_candidates:
    df, meta = safe_read_csv(cand)
    weather_meta.append(meta)
    if not df.empty:
        time_col = next((c for c in df.columns if c.lower() in ["time_utc","time","datetime","timestamp","date"]), None)
        if time_col:
            df["date"] = pd.to_datetime(df[time_col], errors="coerce").dt.normalize()
            keep = {}
            for c in ["wind_speed_ms","wind_dir_deg","cloud_cover_pct","visibility_m","temperature_c",
                      "wind_speed_10m","wind_direction_10m","cloud_cover","visibility","temperature_2m"]:
                if c in df.columns:
                    keep[c] = "mean"
            if keep:
                weather = base_daily.merge(df.groupby("date", as_index=False).agg(keep), on="date", how="left")
                break

ground = pd.DataFrame({"date": dates})
ground_meta = []
for cand in ground_candidates:
    df, meta = safe_read_csv(cand)
    ground_meta.append(meta)
    if not df.empty:
        date_col = next((c for c in df.columns if c.lower() in ["date","day","datetime","timestamp"]), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
            numeric_cols = [c for c in df.columns if c != date_col and pd.api.types.is_numeric_dtype(df[c])]
            agg = {c: "mean" for c in numeric_cols[:8]}
            if agg:
                ground = base_daily.merge(df.groupby("date", as_index=False).agg(agg), on="date", how="left")
                if "ground_rows" not in ground.columns:
                    counts = df.groupby("date").size().rename("ground_rows").reset_index()
                    ground = ground.merge(counts, on="date", how="left")
                break

news = pd.DataFrame({"date": dates, "article_count": 0, "headline_sample": ""})
news_meta = []
for cand in news_candidates:
    p = PROJECT_ROOT / cand
    if p.suffix.lower() == ".json" and p.exists():
        articles = load_json(p)
        rows = []
        for a in articles:
            published = a.get("publishedAt") or a.get("published_at") or a.get("date")
            title = a.get("title","")
            if published:
                dt = pd.to_datetime(published, errors="coerce")
                rows.append({"date": pd.Timestamp(dt).normalize(), "title": title})
        if rows:
            ndf = pd.DataFrame(rows)
            news = base_daily.merge(
                ndf.groupby("date").agg(article_count=("title","size"), headline_sample=("title", lambda x: " | ".join(list(x)[:3]))).reset_index(),
                on="date", how="left"
            ).fillna({"article_count": 0, "headline_sample": ""})
        news_meta.append({"path": str(p), "status": "ok", "sha256": sha256_file(p)})
        break
    else:
        df, meta = safe_read_csv(cand)
        news_meta.append(meta)
        if not df.empty:
            date_col = next((c for c in df.columns if c.lower() in ["date","publishedat","published_at","time","datetime"]), None)
            title_col = next((c for c in df.columns if c.lower() in ["title","headline"]), None)
            if date_col:
                df["date"] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
                if not title_col:
                    df["title"] = ""
                    title_col = "title"
                news = base_daily.merge(
                    df.groupby("date").agg(article_count=(title_col,"size"), headline_sample=(title_col, lambda x: " | ".join([str(v) for v in list(x)[:3]]))).reset_index(),
                    on="date", how="left"
                ).fillna({"article_count": 0, "headline_sample": ""})
                break

daily = base_daily.merge(weather, on="date", how="left").merge(ground, on="date", how="left").merge(news, on="date", how="left")
per_site = []
for _, site in registry.iterrows():
    t = daily.copy()
    t["site_id"] = site.get("id")
    t["site_name"] = site.get("name")
    t["role"] = site.get("role")
    t["lat"] = site.get("lat")
    t["lon"] = site.get("lon")
    t["site_score"] = pd.to_numeric(t.get("article_count", 0), errors="coerce").fillna(0)
    per_site.append(t)

xref = pd.concat(per_site, ignore_index=True) if per_site else daily.copy()
xref.to_csv(out / "cross_reference_daily.csv", index=False)
display(xref.head(20))

manifest = manifest_base(step, [CONFIGS / "run.yml"])
for p in [out / "cross_reference_daily.csv"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["inputs"] = {"registry": registry_meta, "weather": weather_meta, "ground": ground_meta, "news": news_meta}
write_json(out / "manifest.json", manifest)
print("Wrote:", out)